# Análise Quantitativa da Frequência Presencial Semanal do Executivo na Empresa

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import re
import datetime as dt
from datetime import datetime
from datetime import datetime, timedelta
from dateutil import parser
import seaborn as sns
from workalendar.america import Brazil
import calendar
import tabula
import sys
import warnings

pd.set_option('mode.chained_assignment', None)
warnings.filterwarnings("ignore", category=FutureWarning)

#### Consumindo os Dados do arquivo PDF:

In [ ]:
#Caminho do arquivo pdf com os Dados desse colaborador:
caminho1 = R".\relatorio N.P.N.I.  01.24_31.03.pdf"

#Inserindo as tabelas dentro de um array que receberá todas elas:
tables = tabula.read_pdf(caminho1, pages="all", multiple_tables=True, stream=True, relative_area=True, relative_columns=True, area=[8,0,96,100], columns=[9, 32, 36.6, 49, 52, 55, 67, 71, 75, 92])

print(len(tables))

#### Apresentando os valores da primeira tabela:

In [ ]:
# Primeira tabela:
tables[0]

#### Apresentando os valores de todas as tabelas dentro da lista de tabelas, além de numerá-las:

In [ ]:
# Verificando as tabelas lidas:
for i, table in enumerate(tables):
    print(f"Table {i}:")
    display(tables[i])

## 2. Tratamento de Dados:

#### Definição das Funções para Transformação e Tratamento de Dados:
1. **Separar Data e Hora:**  
   Função que recebe uma coluna datetime e separa em duas colunas distintas: uma para data e outra para hora, facilitando análises temporais específicas.

2. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

3. **Remover Caracteres Específicos:**  
   Função para limpar colunas de texto, removendo caracteres especiais e números, mantendo apenas letras e espaços para uniformização dos dados.

4. **Remover Zeros:**  
   Função que elimina ou substitui valores zero em colunas numéricas, prevenindo distorções nas análises quantitativas.

5. **Remover Zeros:**  
   Função que elimina ou substitui valores zero em colunas numéricas, prevenindo distorções nas análises quantitativas.

In [ ]:
# Função para separar a terceira coluna em "Data" e "Hora":
def separar_data_hora(df, col):
    df[['Data', 'Hora']] = df[col].apply(lambda x: pd.Series([x[:10], x[-5:]]))
    df = df.drop(columns=[col])  # Remove a coluna original após a separação...
    return df

# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para remover caracteres de letra:
def remove_letras(df, col):
    # Definir a função para aplicar a cada valor na coluna:
    def apenas_numeros_e_especiais(valor):
        return re.sub(r'[^0-9:/-]', '', str(valor))
    
    df.iloc[:, col] = df.iloc[:, col].apply(apenas_numeros_e_especiais)

    return df

# Função para remover zeros:
def remove_zeros(df, col):
    # Definir a função para aplicar a cada valor na coluna:
    def apenas_zeros_e_especiais(valor):
        return re.sub(r'[^1-9]', '', str(valor))
    
    df.iloc[:, col] = df.iloc[:, col].apply(apenas_zeros_e_especiais)
    
    return df

# Função para remover hiféns:
def remove_hifen(df, col):
    df.iloc[:, col] = df.iloc[:, col].astype(str).str.replace('-', '', regex=False)
    return df


#### Aplicando Funções a lista de tabelas:

In [ ]:
# Aplicando as funções a todas as tabelas:
for i in range(len(tables)):
    print(f"\rProcessando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = remove_zeros(tables[i], 10)
    tables[i] = remove_letras(tables[i], 10)
    tables[i] = remove_letras(tables[i], 3)
    tables[i] = remove_letras(tables[i], 7)
    tables[i] = remove_letras(tables[i], 8)
    tables[i] = separar_data_hora(tables[i], tables[i].columns[3])
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Uso Acesso')
    tables[i] = remove_letras(tables[i], 2)
    tables[i] = remove_hifen(tables[i], 7)

#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso'...


#### Visualizando a primeira tabela após o tratamento dela:

In [ ]:
tables[0]

#### Visualizando todas as tabelas após o tratamento dela:

In [ ]:
# Verificando as tabelas modificadas:
for i, table in enumerate(tables):
    print(f"Table {i}:")
    display(tables[i])

#### Concatenar todas as tabelas da lista em um único Data Frame:

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada
colunas_ordenadas = ['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso']
df_ronda = df_ronda[colunas_ordenadas]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total

# Verificando o DataFrame final
df_ronda

## 3. Analisando os Dados:

#### Formatando tipos de dados:

In [ ]:
# Formatando tipos de dados:
df_ronda.dtypes

#### Descrobrindo todos os valores possíveis de Hora:

In [ ]:
# Obter todas as horas distintas:
horas_distintas = df_ronda['Hora'].unique()

# Exibir as horas distintas:
print(horas_distintas)

#### Descrobrindo todos os valores possíveis de Data:

In [ ]:
# Obter todas as datas distintas:
datas_distintas = df_ronda['Data'].unique()

# Exibir as datas distintas:
print(datas_distintas)

#### Inserindo valores dentro do Data Frame:

In [ ]:
# Data e Hora para o DateTime:
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst='true')
df_ronda['Hora'] = pd.to_datetime(df_ronda['Hora'], format = '%H:%M')

#### Corrigindo o tipo de dado de cada coluna do Data Frame:

In [ ]:
# Formatar categóricos numéricos, como interger, para ficar mais bonito visualmente e formatando todas as outras colunas:
df_ronda['Cadastro'] = df_ronda['Cadastro'].astype(int)
df_ronda['Nome'] = df_ronda['Nome'].astype(str)
df_ronda['Filial'] = df_ronda['Filial'].astype(int)
df_ronda['Dir'] = df_ronda['Dir'].astype(str)
df_ronda['Tipo'] = df_ronda['Tipo'].astype(int)
df_ronda['Acesso'] = df_ronda['Acesso'].astype(str)
df_ronda['Planta'] = df_ronda['Planta'].astype(int)
df_ronda['Coletor'] = df_ronda['Coletor'].astype(int)
df_ronda['Descricao Coletor'] = df_ronda['Descricao Coletor'].astype(str)
df_ronda['Uso Acesso'] = df_ronda['Uso Acesso'].astype(int)

# Demonstrando os tipos de dados que são de cada coluna:
df_ronda.dtypes

#### Visualizando os nomes das colunas:

In [ ]:
# Visualizando os nomes das colunas do Data Frame:
df_ronda.head()

#### Adicionando a coluna 'Dia Semana':

In [ ]:
# Inserindo os dias da semana:
df_ronda['Dia Semana'] = df_ronda['Data'].dt.strftime('%A')
df_ronda

#### Alterando a ordem das colunas:

In [ ]:
# Definir base para realização dos cálculos de presença na fábrica:
df_ronda = df_ronda[['Nome', 'Cadastro', 'Data', 'Hora', 'Dia Semana', 'Dir', 'Tipo', 'Coletor', 'Descricao Coletor', 'Uso Acesso']]
df_ronda

#### Visualizando os nomes e dipos das colunas:

In [ ]:
# Demonstrando os tipos de dados que são de cada coluna:
df_ronda.dtypes

# Integração Analítica Final:
#### Realização da carga final dos dados pós-tratamento na camada analítica (OLAP), visando disponibilização para consultas multidimensionais:

In [ ]:
# Determinar df de entrada presencial - Uso Acesso = 1:
df_presencial = df_ronda[(df_ronda['Uso Acesso'] == 1)]

# Filtrando o primeiro e o último horário de cada dia:
primeiros_horarios = df_presencial.groupby('Data')['Hora'].min().reset_index()
ultimos_horarios = df_presencial.groupby('Data')['Hora'].max().reset_index()

# Mesclando os primeiros e últimos horários de cada dia:
df_presencial_filtrado = pd.merge(primeiros_horarios, ultimos_horarios, on='Data', suffixes=('_primeiro', '_ultimo'))

# Mesclando os registros originais com os primeiros e últimos registros de ponto:
df_presencial_filtrado = pd.merge(df_presencial, df_presencial_filtrado, on='Data')

# Mantendo somente os registros que correspondem ao primeiro ou último horário de cada dia:
df_presencial_filtrado = df_presencial_filtrado[(df_presencial_filtrado['Hora'] == df_presencial_filtrado['Hora_primeiro']) | (df_presencial_filtrado['Hora'] == df_presencial_filtrado['Hora_ultimo'])]

# Removendo colunas desnecessárias:
df_ponto_filtrado = df_presencial_filtrado.drop(columns=['Hora_primeiro', 'Hora_ultimo'])

# Tabela Final:
df_ponto_filtrado

#### Finalizando os tratamentos, mas já dentro do Data Frame final:

In [ ]:
# Criar um df com 'Data Real' distinta - removendo duplicatas para obter a quantidade de dias distintos de presença na planta:
df_datas = df_ponto_filtrado.drop_duplicates(subset = ['Data'])

# df_datas = df_datas[df_datas['Data'].dt.month == 2]:
df_datas

#### Tratando as datas para uma melhor visualização nos gráficos:

In [ ]:
# Fazer um count de 'Dia_Semana' para compreender maior porcentagem de presença por dias:
df_count = df_datas['Dia Semana'].value_counts().reset_index()
df_count.columns = ['Dia Semana', 'Contagem Presença']
df_count

In [ ]:
# Calcular a representativade de 'Contagem_Presença' em porcentagem:
total_presencas = df_count['Contagem Presença'].sum()
df_count['Presença_%'] = (df_count['Contagem Presença']/total_presencas)*100
df_count['Presença_%'] = df_count['Presença_%'].round(2)
df_count

## Criando o primeiro gráfico (Gráfico de Barras):

In [ ]:
# Plotar gráfico de barras com a porcentagem de presença por dia da semana:
plt.figure(figsize=(10, 6))
plt.bar(df_count['Dia Semana'], df_count['Presença_%'])

for i, v in enumerate(df_count['Presença_%']):
    plt.text(i, v, f'{v: .2f}%',
ha='center', va='bottom', fontsize=10, color='black')
    
plt.xlabel('Dia da Semana')
plt.ylabel('Porcentagem de Presença')
plt.title('Porcentagem de Presença por Dia da Semana')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#### Inserindo a coluna para visualizar as datas de maneira mais dinâmica:

In [ ]:
# Criar coluna com semana do ano:
df_datas['Semana Ano'] = df_datas['Data'].dt.isocalendar().week
df_datas

#### Fazendo uma análise entre a presença e os dias durante a semana e ano:

In [ ]:
# Calcular média de presença por semana:
contagem_por_semana = df_datas.groupby('Semana Ano').size()
media_presenca_semana = round(contagem_por_semana.mean(), 2)
media_presenca_semana
print(f"A média de presença por semana do implicado é de {media_presenca_semana}, ou seja, de 5 dias úteis, ele está presencialmente, em média, {media_presenca_semana} dias.")

#### Análise dos dias presenciais por semana:

In [ ]:
df_datas['Ano'] = df_datas['Data'].dt.year
df_datas

#### Fazendo a contagem total da presença por dias:

In [ ]:
# Fazer um count de 'Semana Ano' para compreender maior porcentagem de presença por dias:
df_count_semana = df_datas[['Ano', 'Semana Ano']].value_counts().reset_index()
df_count_semana.columns = ['Ano', 'Semana Ano', 'Contagem Presença']
df_count_semana = df_count_semana.sort_values(by=['Ano', 'Semana Ano']).reset_index(drop=True)
df_count_semana

#### Cria uma tabela somente para o filtro das datas:

In [ ]:
# Define o período:
inicio = datetime(2024, 1, 1)
fim = datetime(2025, 3, 31)

# Gera a lista de datas com frequência semanal (toda segunda-feira):
datas_semanais = pd.date_range(start=inicio, end=fim, freq='W-MON')

# Cria a tabela com início e fim de cada semana:
tabela_semanas = pd.DataFrame({
    'Início da Semana': datas_semanais,
    'Fim da Semana': datas_semanais + timedelta(days=6)
})

# Adiciona a coluna com o número da semana (ISO):
tabela_semanas['Ano'] = tabela_semanas['Início da Semana'].dt.isocalendar().year
tabela_semanas['Semana Ano'] = tabela_semanas['Início da Semana'].dt.isocalendar().week

# Exibe a tabela:
tabela_semanas

#### Continua modulando o filtro por datas:

In [ ]:
# Faz o merge com base em 'Ano' e 'Número da Semana':
df_semana_merged = pd.merge(
    tabela_semanas,
    df_count_semana,
    on=['Ano', 'Semana Ano'],
    how='left'  # Mantém todas as semanas da tabela_semanas...
)

# Preenche valores ausentes com zero:
df_semana_merged['Contagem Presença'] = df_semana_merged['Contagem Presença'].fillna(0).astype(int)

df_semana_merged = df_semana_merged.sort_values(by='Contagem Presença', ascending=False)

# Exibe a tabela final:
df_semana_merged

#### Definir períodos de férias a serem removidos:

In [ ]:
# Datas das férias coletivas:
ferias = [
    (datetime(2023, 12, 18), datetime(2024, 1, 3)),
    (datetime(2024, 12, 23), datetime(2025, 1, 8)),
    (datetime(2024, 9, 23), datetime(2024, 10, 5)),
]

# Função para verificar se uma semana está em um dos períodos de férias:
def esta_em_ferias(inicio_semana):
    for inicio, fim in ferias:
        if inicio <= inicio_semana <= fim:
            return True
    return False

# Filtra semanas que não estão em períodos de férias:
df_semana_merged = df_semana_merged[~df_semana_merged['Início da Semana'].apply(esta_em_ferias)].reset_index(drop=True)

# Exibe a tabela:
df_semana_merged

#### Semanas com somente um registro de dia presencial:

In [ ]:
# Filtra o DataFrame para manter apenas os registros com 'Contagem Presença' igual a 0 ou 1, correspondendo a pessoas com no máximo um registro de presença no dia e reseta o índice para manter a consistência após o filtro:
df_semana_merged_1dia = df_semana_merged[(df_semana_merged['Contagem Presença'] == 1) | (df_semana_merged['Contagem Presença'] == 0)].reset_index(drop=True)

# Remove semana 1 por ser coletivas:
df_semana_merged_1dia = df_semana_merged_1dia[df_semana_merged_1dia['Semana Ano'] != 1].reset_index(drop=True)
df_semana_merged_1dia

## Criando o primeiro gráfico (Gráfico de Pizza):

In [ ]:
# Conta quantas vezes cada valor aparece na coluna 'Contagem Presença':
distribuicao_presencas = df_semana_merged['Contagem Presença'].value_counts().sort_index()

# Define cores: vermelho para 0 e 1, verde para 2 em diante:
cores = []
for presenca in distribuicao_presencas.index:
    if presenca in [0, 1]:
        cores.append('#f28b82')  # vermelho pastel...
    else:
        cores.append('#a5d6a7')  # verde pastel...

# Função personalizada para exibir a quantidade e a porcentagem:
def funcao_autopct(pct, allvals):
    absolute = round(pct / 100.*sum(allvals))
    return f"{absolute}/{sum(allvals)} ({pct:.1f}%)"

# Cria o gráfico de pizza:
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    distribuicao_presencas,
    labels=[f'{int(k)} presença(s)' for k in distribuicao_presencas.index],
    autopct=lambda pct: funcao_autopct(pct, distribuicao_presencas),
    startangle=90,
    counterclock=False,
    labeldistance=1.05,
    colors=cores,
    wedgeprops={'edgecolor': 'black', 'linewidth': 2},
    pctdistance=0.8
)

# Ajuste específico para o rótulo "4 presença(s)":
for i, text in enumerate(texts):
    if '4 presença(s)' in text.get_text():
        text.set_position((0.1, 1.1))  # Ajusta a posição do rótulo "4 presença(s)"...

# Ajusta apenas a posição do valor percentual de "4 presença(s)":
for i, (label, auto) in enumerate(zip(texts, autotexts)):
    if '4 presença(s)' in label.get_text():
        # Pega a posição original e "empurra" mais pra fora:
        x, y = auto.get_position()
        auto.set_position((x * 1.3, y * 1.3))  # Aumenta a distância a partir do centro...

# Título e layout:
ax.set_title('Distribuição da Contagem de Presenças por Semana (janeiro 2024 - março 2025)')
plt.axis('equal')
plt.tight_layout()
plt.show()

#### Carregando os Dados tratados dentro de um Excel:

In [ ]:
# Exportando os Dados:
df_semana_merged.to_excel("presenca_por_semana A.M.N.M. 01.24_31.03.xlsx")